# Rag Pipeline
- Data ingestion to vector DB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Admin\AppData\Local\Temp\ipykernel_21220\329546744.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
j:\DocuBase\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Read all the PDF's

In [2]:
def preprocess_all_pdf(pdf_dir):
    all_docs=[]
    pdf_dir=Path(pdf_dir)

    # find all files
    pdf_files=list(pdf_dir.glob('**/*.pdf'))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader=PyMuPDFLoader(str(pdf_file))
            docs=loader.load()

            # Add source info to metdata
            for doc in docs:
                doc.metadata['source']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_docs.extend(docs)
            print(f"Loaded {len(docs)} pages")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
    
    print(f"Total pages loaded: {len(all_docs)}")
    return all_docs

In [3]:
all_pdf_docs=preprocess_all_pdf('../data')

Found 2 PDF files to process
Processing Machine_Learning_Unit_2.pdf
Loaded 46 pages
Processing Machine_Learning_Unit_3.pdf
Loaded 28 pages
Total pages loaded: 74


In [4]:
all_pdf_docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'TeX', 'creationdate': '2025-09-21T10:02:46+00:00', 'source': 'Machine_Learning_Unit_2.pdf', 'file_path': '..\\data\\Machine_Learning_Unit_2.pdf', 'total_pages': 46, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-12-14T21:03:01+05:30', 'trapped': '', 'modDate': "D:20251214210301+05'30'", 'creationDate': 'D:20250921100246Z', 'page': 0, 'file_type': 'pdf'}, page_content='UNIT 2 Linear and Logistic Regression,\nBayesian Learning, Support Vector Machines\nMLT, BCS 055\n1')

## Chunking: Text splitting

In [5]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks\n")

    if split_docs:
        print("Example of chunks:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}\n")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_docs)

Split 74 documents into 98 chunks

Example of chunks:
Content: UNIT 2 Linear and Logistic Regression,
Bayesian Learning, Support Vector Machines
MLT, BCS 055
1...
Metadata: {'producer': 'pdfTeX-1.40.26', 'creator': 'TeX', 'creationdate': '2025-09-21T10:02:46+00:00', 'source': 'Machine_Learning_Unit_2.pdf', 'file_path': '..\\data\\Machine_Learning_Unit_2.pdf', 'total_pages': 46, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-12-14T21:03:01+05:30', 'trapped': '', 'modDate': "D:20251214210301+05'30'", 'creationDate': 'D:20250921100246Z', 'page': 0, 'file_type': 'pdf'}



## Embedding and VectorDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
# resposible for embedding
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        '''
        initialize the embedding manager
        Args:
            model_name (str): Huggingface transformer model for sentence embeddings.
        '''
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e

    def gen_embeddings(self, texts: List[str]) -> np.ndarray:
        '''
        Generate embeddings for a list of texts.
        Args:
            texts (List[str]): List of text strings to embed.
        Returns:
            np.ndarray: Array of embeddings, with shape of (len(texts), embedding_dimension).
        '''
        if not self.model:
            raise ValueError("Model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [9]:
# initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


j:\DocuBase\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7911.17it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [15]:
# managing documnet embeddings in chromadb vector store
class VectorStore:

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        '''
        Initialize the vector store.
        Args:
            collection_name (str): Name of the collection in ChromaDB.
            persist_directory (str): Directory to persist the ChromaDB data.
        '''
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        #  Initialize ChromaDB client and collection
        try:
            # create persitent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            # print(f"Existing documents in the collection: {len(self.collection.get()['ids'])}")
            print(f"Existing documents in the collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise e

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        '''
        Add documents and their embeddings to the vector store.
        Args:
            documents (List[Any]): List of document objects (e.g., LangChain Document).
            embeddings (np.ndarray): Corresponding embeddings for the documents.
        '''
        if not self.collection:
            raise ValueError("Collection is not initialized.")

        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to the vector store.")

        # Prepare data for ChromaDB
        ids=[]
        metadatas=[]
        docs_text=[]
        embeddings_list=[]

        # Add to collection
        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            # generate unique id for each document
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            # Prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            # doc content
            docs_text.append(doc.page_content)
            # embeddings
            embeddings_list.append(emb.tolist())
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=docs_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise e

In [16]:
vector_store=VectorStore()

Vector store initialized with collection: pdf_documents
Existing documents in the collection: 0


### Converting text to embedding

In [19]:
texts=[doc.page_content for doc in chunks]

# Generate embeddings 
embeddings=embedding_manager.gen_embeddings(texts)

# store embeddings in vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 98 texts.


Batches: 100%|██████████| 4/4 [00:14<00:00,  3.69s/it]


Generated embeddings with shape: (98, 384)
Adding 98 documents to the vector store.
Successfully added 98 documents to the vector store.
Total documents in the collection after addition: 98


## Retriver pipeline from VectorStore

In [48]:
class RAGretriver:
    # Handles query based retrieval from the vector store
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        '''
        Initialize the retriver
        Args:
            vector_store (VectorStore): Instance of the VectorStore class/containing doc embeddings.
            embedding_manager (EmbeddingManager): Instance of the EmbeddingManager class.
        '''
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        '''
        Retrieve top_k relevant documents for the given query.
        Args:
            query (str): The query string.
            top_k (int): Number of top documents to retrieve.
            score_threshold (float): Minimum similarity score for retrieved documents.
        Returns:
            List[Dict[str, Any]]: List of retrieved documents with metadata.
        '''
        print(f"Retrieving for query: {query}")
        print(f"Top_k: {top_k}, Score threshold: {score_threshold}")

        # Generate embedding for the query
        query_embedding=self.embedding_manager.gen_embeddings([query])[0]

        # search in the vector store
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
                # include=['metadatas', 'documents', 'distances']
            )

            # Process results
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                docs=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                # print(f"Raw results - Distances: {distances}")  # Debug: show all distances

                for i, (doc, metadata, distance, doc_id) in enumerate(zip(docs, metadatas, distances, ids)):
                    sim_score=1 - distance  # Convert distance to similarity score

                    # print(f"Doc {i+1}: similarity={sim_score:.4f}, threshold={score_threshold}")  # Debug

                    if sim_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': doc,
                            'metadata': metadata,
                            'sim_score': sim_score,
                            'distance': distance,
                            'Rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents after filtering.")
            else:
                print("No documents found in the vector store for the given query.")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise e

In [49]:
rag_retriever=RAGretriver(vector_store, embedding_manager)
rag_retriever

In [50]:
results = rag_retriever.retrieve("Decision Tree")

Retrieving for query: Decision Tree
Top_k: 5, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.22it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents after filtering.
